**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**05 | Generalisation, Robustness and Final Evaluation**

![Generalisation, Robustness and Final Evaluation — scientific workflow](../../images/05_generalisation_and_final_evaluation.svg)

**Objective:** Reconcile held-out prediction scores, select on Validation, and report matched Test performance and conditional participant uncertainty.

**Dataset:** Fixed trial features, participant assignments and saved scores from Notebooks 03 and 04.

**Scope:** Compare fixed candidates on matched held-out trials, quantify conditional participant uncertainty and examine score calibration without further fitting.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling")

# ---------------------------- 2. Standard Library ----------------------------
import sys

# --------------------------- 3. Study Dependencies ---------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(root / "reading_comprehension_modelling" / "requirements.txt")],
    check=True,
)

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------------ 5. Machine Learning Metrics ------------------------
from sklearn.metrics import confusion_matrix

# ------------------- 6. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 7. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import validate_splits, validate_trial_index
from reading_evaluation import evaluate, participant_bootstrap

# ------------------- 8. Output Directory and Setup Summary -------------------
out_dir = root / "reading_comprehension_modelling" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=10, legend_size=10,
    figure_dpi=100, figure_size=(10, 5),
)
print("Study outputs:", out_dir)
print("Python:", sys.version.split()[0])


**02. Reconcile Matched Held-Out Predictions**

Match the 160,018 saved classical and neural scores to the eligible trial identities, outcome labels and participant roles. Each of the 19 candidates per protocol must cover every assigned Validation and Test trial exactly once. The primary candidate for each protocol is selected by Validation Macro F1, with alphabetical model-name tie-breaking.

TF-IDF fusion is selected in Protocol A (Validation Macro F1 0.5582), and TF-IDF text in Protocol B (0.5698). All other Test comparisons are secondary and do not alter this selection.

In [3]:
# ================ 02. Reconcile Matched Held-Out Predictions =================

# ------------------- 1. Load and Reconcile Trial Features --------------------
identity = ["trial_id","participant_id","trial_index","article_batch",
            "article_key","paragraph_key","question_key","difficulty_level"]
table = pd.read_csv(out_dir/"02_trial_features.csv",dtype={c:str for c in identity})
index = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
index["is_correct"] = index.is_correct.astype(int)
validate_trial_index(index,table)

# ------------------------- 2. Load Participant Roles -------------------------
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
validate_splits(table,splits)

# --------------- 3. Reconcile Scores and Select on Validation ----------------
scores = pd.concat([pd.read_csv(out_dir/"03_classical_predictions.csv"),
                    pd.read_csv(out_dir/"04_neural_predictions.csv")],ignore_index=True)
data, validation, selected, test, people, strata, calibration = evaluate(table,splits,scores)
display(selected[["protocol","model","macro_f1","n_people","n_trials"]])


**03. Compare Test Performance and Generalisation**

Use Macro F1 as the primary metric, complemented by accuracy, balanced accuracy, AUROC, incorrect-response precision, recall and area under the precision–recall curve (AUPRC).

| Metric | A: TF-IDF fusion | B: TF-IDF text |
| --- | ---: | ---: |
| Test participants / trials | 36 / 1,944 | 60 / 3,240 |
| Macro F1 | 0.5560 | 0.5035 |
| Accuracy | 0.6224 | 0.6389 |
| Balanced accuracy | 0.6161 | 0.5116 |
| AUROC | 0.6573 | 0.5386 |
| Incorrect AUPRC | 0.3002 | 0.2050 |
| Incorrect precision | 0.2813 | 0.2032 |
| Incorrect recall | 0.6058 | 0.3058 |

The prevalence reference achieves Macro F1 of 0.4462 and 0.4473 with zero incorrect-response recall. The selected A candidate approximately retains its Validation performance, whereas the selected B candidate shows limited discrimination. Protocol B changes participants and articles together; the difference between protocols is not an isolated unseen-text effect.

TF-IDF fusion attains the highest secondary Test Macro F1 in B (0.5296), but TF-IDF text remains the primary candidate selected on Validation.

In [4]:
# ============== 03. Compare Test Performance and Generalisation ==============

# ----------------------- 1. Summarise Held-Out Metrics -----------------------
display(test.sort_values(["protocol","macro_f1"],ascending=[True,False]))

# ------------------------ 2. Plot Held-Out Comparison ------------------------
fig, axes = plt.subplots(1,2,figsize=(14,8),sharey=True)
for ax,(protocol,g) in zip(axes,test.groupby("protocol")):
    g.sort_values("model").plot.barh(x="model",y="macro_f1",ax=ax,legend=False)
    ax.set(title=protocol,xlabel="Test Macro F1",ylabel="")
fig.tight_layout()
save_figure(fig,out_dir/"05_test_comparison.png")
plt.show()


**04. Analyse Participant Variation and Difficulty**

Summarise Test performance by participant and difficulty. For single-class participant subsets, AUROC, balanced accuracy and incorrect AUPRC are undefined under the implemented metric convention and remain missing rather than being replaced with zero.

Selected-model participant Macro F1 ranges from 0.3448 to 0.6842 (median 0.5261) in A and 0.3721 to 0.7146 (median 0.4893) in B. Advanced/elementary pooled Macro F1 is 0.5606/0.5489 in A and 0.4904/0.4996 in B. Difficulty strata are observational, not causal estimates.

The confusion matrices use observed classes as rows and predicted classes as columns (0 = incorrect, 1 = correct). The selected models identify 229 of 378 incorrect responses in A and 189 of 618 in B.

In [5]:
# ============= 04. Analyse Participant Variation and Difficulty ==============

# -------------- 1. Inspect Participant and Difficulty Summaries --------------
display(people.head())
display(strata)

# ----------------- 2. Plot Selected-Model Confusion Matrices -----------------
primary = test.merge(selected[["protocol","model"]],on=["protocol","model"])
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,row in zip(axes,primary.itertuples(index=False)):
    g=data[(data.protocol==row.protocol)&(data.model==row.model)&data.role.eq("test")]
    cm=confusion_matrix(g.is_correct,g.p_correct>=.5,labels=[0,1])
    ax.imshow(cm,cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    ax.set(xticks=[0,1],yticks=[0,1],xlabel="Predicted",ylabel="Observed",
           title=row.protocol+" / "+row.model)
fig.tight_layout()
save_figure(fig,out_dir/"05_selected_confusion.png")
plt.show()


**05. Quantify Paired Participant Uncertainty**

Resample participant-level confusion counts jointly across candidates (1,000 draws, seed 17). The 95% percentile intervals describe pooled Test Macro F1 and matched modality differences, conditional on the fitted models and held-out participants.

The selected-model intervals are 0.5276–0.5805 in A and 0.4863–0.5211 in B. For TF-IDF fusion minus TF-IDF text, the paired differences are +0.0108 (−0.0127 to +0.0313) in A and +0.0261 (+0.0068 to +0.0432) in B. Logistic Regression and XGBoost fusion also have wholly positive intervals relative to text in B, while Random Forest fusion has wholly negative intervals relative to text in both protocols.

The 14 paired contrasts cover scalar and lexical model families, not neural modality contrasts. They are secondary, conditional and not multiplicity-adjusted. The intervals do not represent article clustering, training variability or external generalisation.

In [6]:
# ================ 05. Quantify Paired Participant Uncertainty ================

# ----------------- 1. Estimate Paired Participant Intervals ------------------
intervals, paired = participant_bootstrap(data,draws=1000,seed=17)
display(intervals)
display(paired)


**06. Assess Score Calibration**

Compare mean predicted score and observed correct-response fraction in ten fixed bins. Marker area reflects the number of trials in each occupied bin; the diagonal represents agreement between predicted score and observed fraction.

The selected-model Brier scores are 0.2272 in A and 0.2220 in B, compared with prevalence-reference scores of 0.1567 and 0.1544. Observed correctness exceeds mean score in eight of nine occupied A bins and all five occupied B bins. Class-weighted scores should therefore not be interpreted as calibrated probabilities. No probability recalibration or Test-driven threshold optimisation is performed.

In [7]:
# ======================= 06. Assess Score Calibration ========================

# ------------------------- 1. Plot Score Calibration -------------------------
fig, axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,row in zip(axes,selected.itertuples(index=False)):
    g=calibration[(calibration.protocol==row.protocol)&(calibration.model==row.model)]
    ax.plot([0,1],[0,1],linestyle="--",color="grey")
    ax.scatter(g.mean_probability,g.observed_fraction,s=20+g.n_trials/g.n_trials.max()*70)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean score",ylabel="Observed correct fraction",
           title=row.protocol)
fig.tight_layout()
save_figure(fig,out_dir/"05_score_calibration.png")
plt.show()


**07. Save Comparative Evidence and Interpret Scope**

Save the eight comparative tables and three figures in `output/`, including Validation selection, Test performance, participant and difficulty metrics, calibration bins and conditional bootstrap intervals.

**Scientific interpretation:** Paragraph-phase gaze provides model-dependent incremental information for predicting the correctness of a subsequent reading-comprehension response. It does not consistently improve on text-only predictors. Ordered neural encoders do not demonstrate a systematic advantage over the classical or lexical candidates; these comparisons do not isolate the contribution of temporal ordering. Generalisation to held-out participants and articles remains limited.

The findings concern ordinary-reading trials within OneStop. They do not establish causal gaze effects, general cognitive ability, clinical validity or external generalisation. Evaluation uses fixed saved predictions; it does not estimate variability across independent model-training runs.

In [8]:
# ============= 07. Save Comparative Evidence and Interpret Scope =============

# --------------------- 1. Save Final Comparative Tables ----------------------
for name,frame in {"05_validation_metrics.csv":validation,"05_selected_models.csv":selected,
                   "05_test_metrics.csv":test,"05_participant_metrics.csv":people,
                   "05_difficulty_metrics.csv":strata,"05_calibration_bins.csv":calibration,
                   "05_participant_intervals.csv":intervals,"05_paired_modality_intervals.csv":paired}.items():
    frame.to_csv(out_dir/name,index=False)

# ------------------------ 2. Package Current Outputs -------------------------
download_outputs_from_colab(out_dir,"05")
